In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
# ── Standardize team names in ALL games_live2 CSV files ──
# Covers filenames AND content columns (away_team_abbrev, home_team_abbrev)

HIST_TO_MODERN = {
    # Relocations / renames
    "SEA": "OKC",   # SuperSonics -> Thunder
    "VAN": "MEM",   # Vancouver Grizzlies -> Memphis
    "NJN": "BKN",   # New Jersey Nets -> Brooklyn
    "NJ":  "BKN",   # New Jersey Nets (short form)
    "BRK": "BKN",   # Brooklyn alt code
    "NOH": "NOP",   # New Orleans Hornets -> Pelicans
    "NOK": "NOP",   # New Orleans/Oklahoma City Hornets -> Pelicans
    "NO":  "NOP",   # New Orleans (short form)

    # Charlotte franchise codes across eras
    "CHH": "CHA",
    "CHO": "CHA",
    "CHA": "CHA",

    # Washington historical codes
    "WSB": "WAS",
    "WSH": "WAS",   # common ESPN-style code
    "BAL": "WAS",
    "WAS": "WAS",

    # Common abbreviation variants (ESPN / thesports style)
    "PHO": "PHX",
    "GS":  "GSW",
    "SA":  "SAS",
    "NY":  "NYK",
    "BK":  "BKN",
    "UTAH": "UTA",

    # Modern 30 teams (identity mappings for completeness)
    "ATL": "ATL", "BOS": "BOS", "BKN": "BKN", "CHI": "CHI", "CLE": "CLE",
    "DAL": "DAL", "DEN": "DEN", "DET": "DET", "GSW": "GSW", "HOU": "HOU",
    "IND": "IND", "LAC": "LAC", "LAL": "LAL", "MEM": "MEM", "MIA": "MIA",
    "MIL": "MIL", "MIN": "MIN", "NOP": "NOP", "NYK": "NYK", "OKC": "OKC",
    "ORL": "ORL", "PHI": "PHI", "PHX": "PHX", "POR": "POR", "SAC": "SAC",
    "SAS": "SAS", "TOR": "TOR", "UTA": "UTA",
}

# All-Star / special event codes to skip (leave as-is)
SPECIAL_TEAMS = {"EAST", "WEST", "USA", "WORLD", "CAN", "DUR", "GIA",
                 "KEN", "LEB", "CHK", "SHQ", "STE"}

def canonical(abbrev):
    """Map any team abbreviation to its modern canonical form."""
    if abbrev is None or (isinstance(abbrev, float) and np.isnan(abbrev)):
        return abbrev
    a = str(abbrev).strip().upper()
    if a in SPECIAL_TEAMS:
        return a  # leave All-Star teams untouched
    return HIST_TO_MODERN.get(a, a)

# Team-related content columns to standardize inside each CSV
TEAM_COLS = ["away_team_abbrev", "home_team_abbrev",
             "away_team_name_alt", "home_team_name_alt"]

games_live2_root = "data/games_live2"
season_dirs = sorted(d for d in os.listdir(games_live2_root)
                     if os.path.isdir(os.path.join(games_live2_root, d)))

files_renamed = 0
files_content_updated = 0
files_total = 0
errors = []

for season_dir in season_dirs:
    season_path = os.path.join(games_live2_root, season_dir)
    csv_files = [f for f in os.listdir(season_path) if f.endswith(".csv")]

    for fname in csv_files:
        files_total += 1
        old_path = os.path.join(season_path, fname)

        # ── 1) Standardize filename ──
        parts = fname.replace(".csv", "").split("_")
        if len(parts) >= 3:
            game_id = parts[0]
            team1 = canonical(parts[1])
            team2 = canonical(parts[2])
            rest = "_".join(parts[3:])  # preserve any extra suffix
            new_fname = f"{game_id}_{team1}_{team2}"
            if rest:
                new_fname += f"_{rest}"
            new_fname += ".csv"
        else:
            new_fname = fname  # unexpected format, leave alone

        new_path = os.path.join(season_path, new_fname)

        # ── 2) Standardize content columns ──
        try:
            df = pd.read_csv(old_path, low_memory=False)
            content_changed = False
            for col in TEAM_COLS:
                if col in df.columns:
                    updated = df[col].apply(lambda v: canonical(v) if pd.notna(v) else v)
                    if not updated.equals(df[col]):
                        df[col] = updated
                        content_changed = True

            # Write to new path (handles rename + content update in one step)
            if content_changed or new_path != old_path:
                df.to_csv(new_path, index=False)
                if content_changed:
                    files_content_updated += 1
                # Remove old file if it was renamed
                if new_path != old_path and os.path.exists(old_path):
                    os.remove(old_path)
                    files_renamed += 1
        except Exception as e:
            errors.append((fname, str(e)))

    print(f"  {season_dir}: {len(csv_files)} files processed")

print(f"\n{'='*60}")
print(f"games_live2 standardization complete")
print(f"  Total files scanned : {files_total}")
print(f"  Filenames renamed   : {files_renamed}")
print(f"  Content updated     : {files_content_updated}")
if errors:
    print(f"  Errors              : {len(errors)}")
    for fn, err in errors[:10]:
        print(f"    {fn}: {err}")
else:
    print(f"  Errors              : 0")

# Quick verification: unique team abbrevs across all filenames
all_teams = set()
for season_dir in season_dirs:
    season_path = os.path.join(games_live2_root, season_dir)
    for fname in os.listdir(season_path):
        if not fname.endswith(".csv"):
            continue
        parts = fname.replace(".csv", "").split("_")
        if len(parts) >= 3:
            all_teams.add(parts[1])
            all_teams.add(parts[2])

non_standard = all_teams - set(HIST_TO_MODERN.values()) - SPECIAL_TEAMS
print(f"\nUnique teams in filenames: {len(all_teams)}")
print(f"Non-standard remaining  : {non_standard if non_standard else 'None (all clean!)'}")

  .ipynb_checkpoints: 1 files processed
  2000: 0 files processed
  2002: 528 files processed
  2003: 1052 files processed
  2004: 1257 files processed
  2005: 1294 files processed
  2006: 1303 files processed
  2007: 1294 files processed
  2008: 1302 files processed
  2009: 1313 files processed
  2010: 1308 files processed
  2011: 1306 files processed
  2012: 1075 files processed
  2013: 1312 files processed
  2014: 1315 files processed
  2015: 1308 files processed
  2016: 1314 files processed
  2017: 1310 files processed
  2018: 1310 files processed
  2019: 1314 files processed
  2020: 1139 files processed
  2021: 1172 files processed
  2022: 1324 files processed
  2023: 1321 files processed
  2024: 1320 files processed
  2025: 1209 files processed
  2026: 816 files processed

games_live2 standardization complete
  Total files scanned : 30517
  Filenames renamed   : 12244
  Content updated     : 30517
  Errors              : 0

Unique teams in filenames: 42
Non-standard remaining  : 

In [2]:
# Load all schedule files to create game_id -> game_date mapping
schedule_files = glob.glob("data/schedules/schedule_*.csv")
print(f"Loading {len(schedule_files)} schedule files...")

game_date_map = {}
for sched_file in schedule_files:
    df = pd.read_csv(sched_file, dtype={"GAME_ID": str})
    for _, row in df.iterrows():
        game_id = row["GAME_ID"].zfill(10)
        game_date_map[game_id] = row["GAME_DATE"]

print(f"Loaded {len(game_date_map)} games with dates")
print(f"Sample: {list(game_date_map.items())[:3]}")

Loading 26 schedule files...
Loaded 30794 games with dates
Sample: [('0020800001', '2008-10-28'), ('0020800003', '2008-10-28'), ('0020800002', '2008-10-28')]


In [3]:
# Load games.csv
games_df = pd.read_csv("data/games.csv", dtype={"game_id": str})
print(f"Loaded {len(games_df)} games from games.csv")
print(f"Sample columns: {list(games_df.columns)[:10]}")

Loaded 30792 games from games.csv
Sample columns: ['game_id', 'game_date', 'season', 'home_team', 'away_team', 'home_team_full', 'away_team_full', 'away_TEAM_CITY', 'away_MIN', 'away_FGM']


In [4]:
# Calculate running win/loss records for each team within each season
# For each game, look at all PREVIOUS games in the same season to compute records
# (no future leakage, no cross-season bleed)

print("Computing running win/loss records per season...")

# Ensure sorted by date
games_df['game_date'] = pd.to_datetime(games_df['game_date'])
games_df = games_df.sort_values('game_date').reset_index(drop=True)

# Initialize new columns
games_df['home_wins'] = 0
games_df['home_losses'] = 0
games_df['away_wins'] = 0
games_df['away_losses'] = 0

# Track records per season per team: {season: {team: {'wins': int, 'losses': int}}}
season_records = {}

for idx, row in games_df.iterrows():
    season = row['season']
    home_team = row['home_team']
    away_team = row['away_team']
    
    # Initialize season tracking if needed
    if season not in season_records:
        season_records[season] = {}
    
    if home_team not in season_records[season]:
        season_records[season][home_team] = {'wins': 0, 'losses': 0}
    
    if away_team not in season_records[season]:
        season_records[season][away_team] = {'wins': 0, 'losses': 0}
    
    # Set the current record BEFORE this game (no leakage)
    games_df.at[idx, 'home_wins'] = season_records[season][home_team]['wins']
    games_df.at[idx, 'home_losses'] = season_records[season][home_team]['losses']
    games_df.at[idx, 'away_wins'] = season_records[season][away_team]['wins']
    games_df.at[idx, 'away_losses'] = season_records[season][away_team]['losses']
    
    # Determine who won this game by comparing points, then update records
    home_pts = row['home_PTS']
    away_pts = row['away_PTS']
    
    if pd.notna(home_pts) and pd.notna(away_pts):
        if float(home_pts) > float(away_pts):
            # Home team won
            season_records[season][home_team]['wins'] += 1
            season_records[season][away_team]['losses'] += 1
        elif float(away_pts) > float(home_pts):
            # Away team won
            season_records[season][away_team]['wins'] += 1
            season_records[season][home_team]['losses'] += 1
    
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(games_df)} games...")

print(f"\nDone! Added columns: home_wins, home_losses, away_wins, away_losses")
print(f"Seasons found: {sorted(season_records.keys())}")

# Show sample of mid-season games so records are non-zero
mid_season = games_df[(games_df['home_wins'] + games_df['home_losses'] > 10)].head(10)
print(f"\nSample records (mid-season games):")
print(mid_season[['game_date', 'season', 'home_team', 'away_team', 'home_PTS', 'away_PTS', 
                   'home_wins', 'home_losses', 'away_wins', 'away_losses']].to_string(index=False))

Computing running win/loss records per season...
Processed 5000/30792 games...
Processed 10000/30792 games...
Processed 15000/30792 games...
Processed 20000/30792 games...
Processed 25000/30792 games...
Processed 30000/30792 games...

Done! Added columns: home_wins, home_losses, away_wins, away_losses
Seasons found: ['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

Sample records (mid-season games):
 game_date  season home_team away_team  home_PTS  away_PTS  home_wins  home_losses  away_wins  away_losses
2000-11-20 2000-01       LAC       BKN        85        86          4            7          5            4
2000-11-21 2000-01       DAL       OKC       110       116          5            7          5            6
2000-11-21 2000-01       WAS       POR

In [5]:
# Save the dataframe with win/loss records back to training_games.csv
output_file = "data/training_games.csv"
games_df.to_csv(output_file, index=False)
print(f"✓ Saved {len(games_df)} games with win/loss records to {output_file}")
print(f"New columns added: home_wins, home_losses, away_wins, away_losses")

✓ Saved 30792 games with win/loss records to data/training_games.csv
New columns added: home_wins, home_losses, away_wins, away_losses


In [6]:
training_games2 = games_df.copy()
training_games2['away_eFG'] = training_games2['away_FGM'] + 0.5 * training_games2['away_FG3M'] / training_games2['away_FGA'] * 100
training_games2['home_eFG'] = training_games2['home_FGM'] + 0.5 * training_games2['away_FG3M'] / training_games2['home_FGA'] * 100

In [7]:
def convert_minutes_to_decimal(minutes_str):
    """Convert MM:SS format to decimal minutes."""
    if pd.isna(minutes_str):
        return 0.0
    
    minutes_str = str(minutes_str).strip()
    
    # If already a number, return it
    try:
        return float(minutes_str)
    except ValueError:
        pass
    
    # Parse MM:SS format
    if ':' in minutes_str:
        parts = minutes_str.split(':')
        if len(parts) == 2:
            try:
                mins = int(parts[0])
                secs = int(parts[1])
                return float(mins + (secs / 60.0))
            except ValueError:
                return 0.0
    
    return 0.0

# Test the function
print(f"Test: '22:07' -> {convert_minutes_to_decimal('22:07'):.2f} minutes")
print(f"Test: '15:30' -> {convert_minutes_to_decimal('15:30'):.2f} minutes")
print(f"Test: '3:45' -> {convert_minutes_to_decimal('3:45'):.2f} minutes")
print(f"Test: 25.5 -> {convert_minutes_to_decimal(25.5):.2f} minutes")

Test: '22:07' -> 22.12 minutes
Test: '15:30' -> 15.50 minutes
Test: '3:45' -> 3.75 minutes
Test: 25.5 -> 25.50 minutes


In [8]:
# Get all player game files to create a mapping from game_id to file
player_game_files = glob.glob("data/players_live/*.csv")
game_id_to_file = {}

for game_file in player_game_files:
    filename = os.path.basename(game_file)
    game_id = filename.split("_")[0]
    game_id_to_file[game_id] = game_file

print(f"Found {len(game_id_to_file)} player game files")
print(f"Sample game_ids: {list(game_id_to_file.keys())[:5]}")

Found 30792 player game files
Sample game_ids: ['0020500912', '0021500858', '0021600214', '0020100175', '0021200373']


In [9]:
# Define specific stats to compute recency-weighted averages for
stat_names = ['FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
              'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']

# Recency weighting config
DECAY = 0.9       # Exponential decay factor (0.9^t, t=1 most recent -> highest weight)
MIN_GAMES = 4     # Minimum games played in current season before including in dataset

# Track each team's game history PER SEASON (no cross-season bleed)
# Key: (team_abbr, season) -> list of stat dicts in chronological order
team_season_history = {}

# Store rows that have sufficient season history
training_rows = []
skipped_insufficient_history = 0

def compute_weighted_stats(history, stat_names, decay):
    """
    Compute recency-weighted averages for a team's season history.
    history: list of stat dicts in chronological order (oldest first)
    Returns dict of {stat_name: weighted_average}
    
    Weighting: most recent game gets 0.9^1, second most recent gets 0.9^2, etc.
    All weights are normalized to sum to 1.
    """
    n = len(history)
    # Reverse so index 0 = most recent game
    games_reversed = list(reversed(history))
    
    # Raw weights: 0.9^1, 0.9^2, ..., 0.9^n
    raw_weights = [decay ** (t + 1) for t in range(n)]
    weight_total = sum(raw_weights)
    norm_weights = [w / weight_total for w in raw_weights]
    
    result = {}
    for stat in stat_names:
        weighted_sum = 0.0
        weight_used = 0.0
        for i, game in enumerate(games_reversed):
            val = game.get(stat)
            if val is not None and not np.isnan(val):
                weighted_sum += norm_weights[i] * val
                weight_used += norm_weights[i]
        
        if weight_used > 0:
            # Re-normalize in case some games had NaN for this stat
            result[stat] = weighted_sum / weight_used
        else:
            result[stat] = np.nan
    
    return result

for idx, row in training_games2.iterrows():
    away_team = row['away_team']
    home_team = row['home_team']
    season = row['season']
    
    away_key = (away_team, season)
    home_key = (home_team, season)
    
    # Get season-specific history (only games from this season)
    away_history = team_season_history.get(away_key, [])
    home_history = team_season_history.get(home_key, [])
    
    if len(away_history) < MIN_GAMES or len(home_history) < MIN_GAMES:
        # Not enough season history yet, skip this game
        skipped_insufficient_history += 1
    else:
        # Start with all original data
        new_row = row.to_dict()
        
        # Store original PTS values before we replace them
        original_away_PTS = new_row.get('away_PTS')
        original_home_PTS = new_row.get('home_PTS')
        
        # Compute recency-weighted stats for away team (season only)
        away_weighted = compute_weighted_stats(away_history, stat_names, DECAY)
        for stat in stat_names:
            new_row[f'away_{stat}'] = away_weighted[stat]
        
        # Compute recency-weighted stats for home team (season only)
        home_weighted = compute_weighted_stats(home_history, stat_names, DECAY)
        for stat in stat_names:
            new_row[f'home_{stat}'] = home_weighted[stat]
        
        # Add back the original PTS columns
        new_row['away_PTS_actual'] = original_away_PTS
        new_row['home_PTS_actual'] = original_home_PTS
        
        training_rows.append(new_row)
    
    # Update team's season history with current game stats (for future games)
    # Away team stats
    if away_key not in team_season_history:
        team_season_history[away_key] = []
    
    away_game_stats = {}
    for stat in stat_names:
        away_col = f'away_{stat}'
        if away_col in row.index:
            val = row[away_col]
            try:
                away_game_stats[stat] = float(val) if not pd.isna(val) else np.nan
            except (ValueError, TypeError):
                away_game_stats[stat] = np.nan
    team_season_history[away_key].append(away_game_stats)
    
    # Home team stats
    if home_key not in team_season_history:
        team_season_history[home_key] = []
    
    home_game_stats = {}
    for stat in stat_names:
        home_col = f'home_{stat}'
        if home_col in row.index:
            val = row[home_col]
            try:
                home_game_stats[stat] = float(val) if not pd.isna(val) else np.nan
            except (ValueError, TypeError):
                home_game_stats[stat] = np.nan
    team_season_history[home_key].append(home_game_stats)
    
    # Progress indicator
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(training_games2)} games, {len(training_rows)} included...")

print(f"\n{'='*60}")
print(f"Processing complete:")
print(f"  Total games processed: {len(training_games2)}")
print(f"  Games included ({MIN_GAMES}+ season games for both teams): {len(training_rows)}")
print(f"  Games dropped (insufficient season history): {skipped_insufficient_history}")
print(f"  Decay factor: {DECAY} (most recent game weight: {DECAY:.1f}/total)")


Processed 5000/30792 games, 4686 included...
Processed 10000/30792 games, 9427 included...
Processed 15000/30792 games, 14169 included...
Processed 20000/30792 games, 18913 included...
Processed 25000/30792 games, 23658 included...
Processed 30000/30792 games, 28405 included...

Processing complete:
  Total games processed: 30792
  Games included (4+ season games for both teams): 29134
  Games dropped (insufficient season history): 1658
  Decay factor: 0.9 (most recent game weight: 0.9/total)


In [10]:
# Convert to DataFrame and save
training_games3 = pd.DataFrame(training_rows)
print(f"Columns: {len(training_games3.columns)}")

Columns: 57


In [11]:
# Get all player stats files
player_game_files = glob.glob("data/players_live/*.csv")
print(f"Found {len(player_game_files)} player game files to process")

# Track stats
games_processed = 0
players_updated = set()
games_without_date = []

# Process each game file
for game_file in player_game_files:
    # Extract game_id from filename (format: GAME_ID_AWAY_HOME.csv)
    filename = os.path.basename(game_file)
    game_id = filename.split("_")[0]
    
    # Look up game date
    if game_id not in game_date_map:
        games_without_date.append(game_id)
        continue
    
    game_date = game_date_map[game_id]
    
    # Load player stats for this game
    try:
        game_df = pd.read_csv(game_file, dtype={"personId": str, "PLAYER_ID": str})
    except Exception as e:
        print(f"Error reading {game_file}: {e}")
        continue
    
    if game_df.empty:
        continue
    
    # Determine player_id column name (could be personId or PLAYER_ID)
    player_id_col = None
    if "player_id" in game_df.columns:
        player_id_col = "player_id"
    elif "PLAYER_ID" in game_df.columns:
        player_id_col = "PLAYER_ID"
    else:
        print(f"Warning: No player ID column found in {game_file}")
        continue
    
    # Add game metadata to each row
    game_df["GAME_ID"] = game_id
    game_df["GAME_DATE"] = game_date
    
    # Process each player in this game
    for _, player_row in game_df.iterrows():
        player_id = str(player_row[player_id_col])
        
        if pd.isna(player_id) or player_id == "" or player_id == "nan":
            continue
        
        # Create/append to player's CSV file
        player_file = f"data/players_stats/{player_id}.csv"
        
        # Convert row to DataFrame for appending
        row_df = pd.DataFrame([player_row])
        
        if os.path.exists(player_file):
            # Append to existing file
            row_df.to_csv(player_file, mode='a', header=False, index=False)
        else:
            # Create new file with header
            row_df.to_csv(player_file, mode='w', header=True, index=False)
        
        players_updated.add(player_id)
    
    games_processed += 1
    
    # Progress indicator every 1000 games
    if games_processed % 1000 == 0:
        print(f"Processed {games_processed} games, {len(players_updated)} unique players so far...")

print(f"\n{'='*60}")
print(f"Summary:")
print(f"  Games processed: {games_processed}")
print(f"  Unique players: {len(players_updated)}")
print(f"  Games without date mapping: {len(games_without_date)}")

if games_without_date:
    print(f"\nSample games without dates: {games_without_date[:5]}")

Found 30792 player game files to process
Processed 1000 games, 2707 unique players so far...
Processed 2000 games, 2979 unique players so far...
Processed 3000 games, 3088 unique players so far...
Processed 4000 games, 3142 unique players so far...
Processed 5000 games, 3179 unique players so far...
Processed 6000 games, 3202 unique players so far...
Processed 7000 games, 3220 unique players so far...
Processed 8000 games, 3236 unique players so far...
Processed 9000 games, 3247 unique players so far...
Processed 10000 games, 3258 unique players so far...
Processed 11000 games, 3264 unique players so far...
Processed 12000 games, 3268 unique players so far...
Processed 13000 games, 3274 unique players so far...
Processed 14000 games, 3281 unique players so far...
Processed 15000 games, 3287 unique players so far...
Processed 16000 games, 3290 unique players so far...
Processed 17000 games, 3292 unique players so far...
Processed 18000 games, 3294 unique players so far...
Processed 1900

In [12]:
# Load teams.csv to map team_id to abbreviation
# Load teams.csv for team_id -> abbreviation mapping
teams_df = pd.read_csv("data/teams.csv", dtype={"team_id": str})
team_id_to_abbr = dict(zip(teams_df["team_id"], teams_df["abbreviation"]))
print(f"Loaded {len(team_id_to_abbr)} team mappings")

# Process each game and add weighted player info by home/away team
MIN_MINUTES = 4.0  # Filter out players with < 4 minutes

training_rows = []
games_with_players = 0
games_without_players = 0

for idx, game_row in training_games3.iterrows():
    game_id = str(game_row["game_id"]).zfill(10)
    
    # Start with all game data
    training_row = game_row.to_dict()
    
    # Get home and away team abbreviations from games.csv
    home_team_abbr = game_row["home_team"]
    away_team_abbr = game_row["away_team"]
    
    # Look up player stats for this game
    if game_id not in game_id_to_file:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Load player stats
    try:
        players_df = pd.read_csv(
            game_id_to_file[game_id], 
            dtype={"personId": str, "PLAYER_ID": str, "teamId": str, "TEAM_ID": str}
        )
    except Exception as e:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    if players_df.empty:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Determine column names
    player_id_col = "personId" if "personId" in players_df.columns else ("PLAYER_ID" if "PLAYER_ID" in players_df.columns else None)
    team_id_col = "teamId" if "teamId" in players_df.columns else ("TEAM_ID" if "TEAM_ID" in players_df.columns else None)
    
    # Try to find minutes column
    minutes_col = None
    for col in ["minutes", "MIN", "min"]:
        if col in players_df.columns:
            minutes_col = col
            break
    
    if not minutes_col or not player_id_col or not team_id_col:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Convert minutes from MM:SS to decimal
    players_df['minutes_decimal'] = players_df[minutes_col].apply(convert_minutes_to_decimal)
    
    # Filter players by minutes (>= 4 minutes)
    active_players = players_df[players_df['minutes_decimal'] >= MIN_MINUTES].copy()
    
    if active_players.empty:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Get unique teams (should be 2)
    unique_teams = active_players[team_id_col].dropna().unique()
    
    if len(unique_teams) < 2:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Map team_ids to abbreviations to determine which is home/away
    home_team_id = None
    away_team_id = None
    
    for team_id in unique_teams:
        team_id_str = str(team_id)
        team_abbr = team_id_to_abbr.get(team_id_str, None)
        
        if team_abbr == home_team_abbr:
            home_team_id = team_id_str
        elif team_abbr == away_team_abbr:
            away_team_id = team_id_str
    
    # Skip if we can't match both teams
    if not home_team_id or not away_team_id:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Process away team players
    away_players = active_players[active_players[team_id_col].astype(str) == away_team_id].copy()
    away_total_minutes = float(away_players['minutes_decimal'].sum())
    
    if away_total_minutes > 0:
        # Calculate weight using the decimal minutes column
        away_players["weight"] = away_players['minutes_decimal'].astype(float) / away_total_minutes
        
        for i, (_, player) in enumerate(away_players.iterrows(), 1):
            player_id = str(player[player_id_col])
            weight = float(player["weight"])
            training_row[f"away_player_{i}_id"] = player_id
            training_row[f"away_player_{i}_weight"] = weight
    
    # Process home team players
    home_players = active_players[active_players[team_id_col].astype(str) == home_team_id].copy()
    home_total_minutes = float(home_players['minutes_decimal'].sum())
    
    if home_total_minutes > 0:
        # Calculate weight using the decimal minutes column
        home_players["weight"] = home_players['minutes_decimal'].astype(float) / home_total_minutes
        
        for i, (_, player) in enumerate(home_players.iterrows(), 1):
            player_id = str(player[player_id_col])
            weight = float(player["weight"])
            training_row[f"home_player_{i}_id"] = player_id
            training_row[f"home_player_{i}_weight"] = weight
    
    training_rows.append(training_row)
    games_with_players += 1
    
    # Progress indicator
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(training_games3)} games, {games_with_players} with players...")

print(f"\n{'='*60}")
print(f"Processing complete:")
print(f"  Games with player data: {games_with_players}")
print(f"  Games without player data: {games_without_players}")
print(f"  Total: {len(training_rows)}")

Loaded 30 team mappings
Processed 5000/29134 games, 5000 with players...
Processed 10000/29134 games, 10000 with players...
Processed 15000/29134 games, 15000 with players...
Processed 20000/29134 games, 20000 with players...
Processed 25000/29134 games, 25000 with players...

Processing complete:
  Games with player data: 28429
  Games without player data: 705
  Total: 29134


In [13]:
training_games2 = pd.DataFrame(training_rows)
print(f"Columns: {len(training_games2.columns)}")

Columns: 117


In [14]:
# Save to CSV
output_file = "data/training_games2.csv"
training_games2.to_csv(output_file, index=False)
print(f"\n✓ Saved {len(training_games2)} training games to {output_file}")


✓ Saved 29134 training games to data/training_games2.csv


In [15]:
# Load training_games3.csv
print("Loading training_games3.csv...")
training_games3 = pd.read_csv("data/training_games2.csv", dtype={"game_id": str})

Loading training_games3.csv...


In [19]:
PLAYER_STAT_COLS = ['FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
                    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']

WINDOWS = [5, 10]  # 5-game and 10-game rolling windows

# Use training_games3 (has player roster columns)
df_src = training_games3.copy()
print(f"Source games: {len(df_src)}")

# Check if output file exists and load already processed game_ids
output_file = "data/training_games3.csv"
processed_game_ids = set()

if os.path.exists(output_file):
    print(f"Found existing {output_file}, loading processed games...")
    df_existing = pd.read_csv(output_file, dtype={"game_id": str})
    processed_game_ids = set(df_existing['game_id'].astype(str).values)
    print(f"Already processed: {len(processed_game_ids)} games")
    del df_existing
else:
    print(f"{output_file} not found, starting fresh")

# Filter out already processed games
df_games_to_process = df_src[~df_src['game_id'].astype(str).isin(processed_game_ids)].copy()
print(f"Games remaining to process: {len(df_games_to_process)}")

if len(df_games_to_process) == 0:
    print("\n✓ All games already processed!")
else:
    # Cache for loaded player stats
    player_stats_cache = {}

    def load_player_stats(player_id):
        """Load player stats CSV, return DataFrame or None."""
        if player_id in player_stats_cache:
            return player_stats_cache[player_id]
        try:
            player_id_int = int(float(player_id))
        except (ValueError, TypeError):
            player_stats_cache[player_id] = None
            return None
        player_file = f"data/players_stats/{player_id_int}.csv"
        if not os.path.exists(player_file):
            player_stats_cache[player_id] = None
            return None
        try:
            df = pd.read_csv(player_file, dtype={"GAME_ID": str, "game_id": str})
            player_stats_cache[player_id] = df
            return df
        except Exception:
            player_stats_cache[player_id] = None
            return None

    def convert_min_to_numeric(min_str):
        """Convert MIN from MM:SS or numeric to float minutes."""
        if pd.isna(min_str):
            return 0.0
        try:
            return float(min_str)
        except (ValueError, TypeError):
            pass
        try:
            min_str = str(min_str).strip()
            if ':' in min_str:
                parts = min_str.split(':')
                return float(parts[0]) + (float(parts[1]) / 60.0 if len(parts) > 1 else 0.0)
            return float(min_str)
        except:
            return 0.0

    def get_player_history(player_id, current_game_id, current_game_date, max_window=10):
        """
        Get player's recent game stats (excluding current game).
        Returns dict with keys for each window size:
          {5: {stat: mean, ...}, 10: {stat: mean, ...}}
        Returns None if player has zero prior games.
        """
        df_player = load_player_stats(player_id)
        if df_player is None or df_player.empty:
            return None
        
        game_id_col = "GAME_ID" if "GAME_ID" in df_player.columns else ("game_id" if "game_id" in df_player.columns else None)
        game_date_col = "GAME_DATE" if "GAME_DATE" in df_player.columns else ("game_date" if "game_date" in df_player.columns else None)
        if not game_id_col or not game_date_col:
            return None
        
        df_p = df_player.copy()
        
        # Filter: only games where player actually played (MIN > 0)
        if "MIN" in df_p.columns:
            df_p['min_numeric'] = df_p['MIN'].apply(convert_min_to_numeric)
            df_p = df_p[df_p['min_numeric'] > 0]
        
        # Exclude current game (no leakage)
        df_p = df_p[df_p[game_id_col].astype(str) != str(current_game_id)]
        if df_p.empty:
            return None
        
        # Sort by date descending, take up to max_window recent games
        df_p[game_date_col] = pd.to_datetime(df_p[game_date_col], errors='coerce')
        df_p = df_p.dropna(subset=[game_date_col])
        # Only include games BEFORE current game date
        current_dt = pd.to_datetime(current_game_date, errors='coerce')
        if current_dt is not None and not pd.isna(current_dt):
            df_p = df_p[df_p[game_date_col] < current_dt]
        df_p = df_p.sort_values(game_date_col, ascending=False).head(max_window)
        if df_p.empty:
            return None
        
        # Compute means for each window size
        result = {}
        for w in WINDOWS:
            window_df = df_p.head(w)  # top-w most recent
            stats = {}
            for stat in PLAYER_STAT_COLS:
                if stat in window_df.columns:
                    vals = pd.to_numeric(window_df[stat], errors='coerce').dropna()
                    stats[stat] = vals.mean() if len(vals) > 0 else 0.0
                else:
                    stats[stat] = 0.0
            result[w] = stats
        
        return result

    def save_checkpoint(output_rows, output_file, mode='a'):
        """Save current batch to CSV."""
        if not output_rows:
            return
        df_batch = pd.DataFrame(output_rows)
        if mode == 'w' or not os.path.exists(output_file):
            df_batch.to_csv(output_file, index=False, mode='w')
        else:
            df_batch.to_csv(output_file, index=False, mode='a', header=False)
        print(f"  ✓ Checkpoint: {len(output_rows)} games written to {output_file}")

    print("\nProcessing games: computing 5-game and 10-game weighted player averages...")
    print("Saving checkpoint every 5000 games...\n")

    output_rows = []
    total_processed = 0
    dropped_no_data = 0
    
    for idx, row in df_games_to_process.iterrows():
        game_id = str(row['game_id'])
        game_date = row['game_date']
        
        # Copy all non-player-roster columns
        new_row = {}
        for col in df_src.columns:
            if not (col.startswith('away_player_') or col.startswith('home_player_')):
                new_row[col] = row[col]
        
        # Initialize aggregated stats for each window: {window: {stat: 0.0}}
        away_agg = {w: {stat: 0.0 for stat in PLAYER_STAT_COLS} for w in WINDOWS}
        home_agg = {w: {stat: 0.0 for stat in PLAYER_STAT_COLS} for w in WINDOWS}
        
        has_away_data = False
        has_home_data = False
        
        # Process away team players
        for i in range(1, 16):
            pid_col = f'away_player_{i}_id'
            wt_col = f'away_player_{i}_weight'
            if pid_col not in row.index or pd.isna(row[pid_col]):
                continue
            player_id = row[pid_col]
            weight = row[wt_col] if wt_col in row.index and not pd.isna(row[wt_col]) else 0.0
            if weight == 0.0:
                continue
            
            history = get_player_history(player_id, game_id, game_date, max_window=10)
            if history is None:
                continue  # no prior data for this player, skip their contribution
            
            has_away_data = True
            for w in WINDOWS:
                for stat in PLAYER_STAT_COLS:
                    away_agg[w][stat] += weight * history[w][stat]
        
        # Process home team players
        for i in range(1, 16):
            pid_col = f'home_player_{i}_id'
            wt_col = f'home_player_{i}_weight'
            if pid_col not in row.index or pd.isna(row[pid_col]):
                continue
            player_id = row[pid_col]
            weight = row[wt_col] if wt_col in row.index and not pd.isna(row[wt_col]) else 0.0
            if weight == 0.0:
                continue
            
            history = get_player_history(player_id, game_id, game_date, max_window=10)
            if history is None:
                continue
            
            has_home_data = True
            for w in WINDOWS:
                for stat in PLAYER_STAT_COLS:
                    home_agg[w][stat] += weight * history[w][stat]
        
        # Drop the game if neither team has any player history
        if not has_away_data and not has_home_data:
            dropped_no_data += 1
            total_processed += 1
            if total_processed % 1000 == 0:
                print(f"Processed {total_processed}/{len(df_games_to_process)} games...")
            continue
        
        # Add aggregated player stats for each window
        for w in WINDOWS:
            for stat in PLAYER_STAT_COLS:
                new_row[f'away_{w}g_player_{stat}'] = away_agg[w][stat]
                new_row[f'home_{w}g_player_{stat}'] = home_agg[w][stat]
        
        output_rows.append(new_row)
        total_processed += 1
        
        if len(output_rows) >= 5000:
            mode = 'w' if not os.path.exists(output_file) and total_processed - dropped_no_data == len(output_rows) else 'a'
            save_checkpoint(output_rows, output_file, mode=mode)
            output_rows = []
        
        if total_processed % 1000 == 0:
            print(f"Processed {total_processed}/{len(df_games_to_process)} games, {dropped_no_data} dropped...")

    if output_rows:
        mode = 'w' if not os.path.exists(output_file) else 'a'
        save_checkpoint(output_rows, output_file, mode=mode)

    print(f"\n{'='*60}")
    print(f"Processing complete!")
    print(f"  Total games processed: {total_processed}")
    print(f"  Games kept: {total_processed - dropped_no_data}")
    print(f"  Games dropped (no player history): {dropped_no_data}")
    print(f"\n✓ Saved to {output_file}")


Source games: 29134
Found existing data/training_games3.csv, loading processed games...
Already processed: 25000 games
Games remaining to process: 4134

Processing games: computing 5-game and 10-game weighted player averages...
Saving checkpoint every 5000 games...

Processed 1000/4134 games, 0 dropped...
Processed 2000/4134 games, 0 dropped...
Processed 3000/4134 games, 0 dropped...
Processed 4000/4134 games...
  ✓ Checkpoint: 3429 games written to data/training_games3.csv

Processing complete!
  Total games processed: 4134
  Games kept: 3429
  Games dropped (no player history): 705

✓ Saved to data/training_games3.csv


In [20]:
# Verify training_games3.csv structure
print("Verifying training_games3.csv...")
df_verify = pd.read_csv("data/training_games3.csv", dtype={"game_id": str})

print(f"\\nShape: {df_verify.shape}")
print(f"Total columns: {len(df_verify.columns)}")

# Check for player roster columns (should be gone)
roster_cols = [col for col in df_verify.columns if ('_player_' in col and ('_id' in col or '_weight' in col))]
print(f"\\nRoster columns (should be 0): {len(roster_cols)}")
if roster_cols:
    print(f"  Found: {roster_cols[:5]}")

# Check for new aggregated player stat columns
away_player_stat_cols = [col for col in df_verify.columns if col.startswith('away_player_') and not col.endswith('_id') and not col.endswith('_weight')]
home_player_stat_cols = [col for col in df_verify.columns if col.startswith('home_player_') and not col.endswith('_id') and not col.endswith('_weight')]

print(f"\\nAway player stat columns: {len(away_player_stat_cols)}")
print(f"  {away_player_stat_cols}")

print(f"\\nHome player stat columns: {len(home_player_stat_cols)}")
print(f"  {home_player_stat_cols}")

# Check for preserved columns
print(f"\\naway_PTS_actual present: {'away_PTS_actual' in df_verify.columns}")
print(f"home_PTS_actual present: {'home_PTS_actual' in df_verify.columns}")

# Show sample row
print(f"\\nSample game:")
sample = df_verify.iloc[0]
print(f"Game ID: {sample['game_id']}")
print(f"Date: {sample['game_date']}")
print(f"Teams: {sample['away_team']} @ {sample['home_team']}")
print(f"\\nAway player stats (sample):")
for stat in ['FGA', 'FG_PCT', 'PTS', 'AST', 'REB']:
    col = f'away_player_{stat}'
    if col in sample:
        print(f"  {col}: {sample[col]:.4f}")
print(f"\\nHome player stats (sample):")
for stat in ['FGA', 'FG_PCT', 'PTS', 'AST', 'REB']:
    col = f'home_player_{stat}'
    if col in sample:
        print(f"  {col}: {sample[col]:.4f}")

print(f"\\nActual points:")
print(f"  Away: {sample['away_PTS_actual']}")
print(f"  Home: {sample['home_PTS_actual']}")

Verifying training_games3.csv...
\nShape: (28429, 129)
Total columns: 129
\nRoster columns (should be 0): 0
\nAway player stat columns: 0
  []
\nHome player stat columns: 0
  []
\naway_PTS_actual present: True
home_PTS_actual present: True
\nSample game:
Game ID: 0020000058
Date: 2000-11-07
Teams: LAL @ HOU
\nAway player stats (sample):
\nHome player stats (sample):
\nActual points:
  Away: 74
  Home: 84


In [13]:
nba_training_df = pd.read_csv('data/training_games3.csv')

In [14]:
nba_training_df.drop(columns=['season', 'home_team_full', 'away_team_full', 'away_TEAM_CITY', 'home_TEAM_CITY'], inplace=True)
nba_training_df.head()

,game_id,game_date,home_team,away_team,away_MIN,away_FGM,away_FGA,away_FG_PCT,away_FG3M,away_FG3A,...,away_10g_player_BLK,home_10g_player_BLK,away_10g_player_TO,home_10g_player_TO,away_10g_player_PF,home_10g_player_PF,away_10g_player_PTS,home_10g_player_PTS,away_10g_player_PLUS_MINUS,home_10g_player_PLUS_MINUS
0,20000058,2000-11-07,HOU,LAL,240:00,34.959581,80.359116,0.434748,4.869439,14.495202,...,1.045914,0.458521,1.713223,1.612160,2.490622,2.672306,15.149697,11.308576,5.335229,1.380674
1,20000060,2000-11-07,POR,ATL,240:00,33.588252,79.188427,0.426121,4.781623,12.173597,...,0.654998,0.850657,2.501213,2.123538,3.193895,2.929524,11.575277,10.350852,-7.701101,-4.093827
2,20000067,2000-11-08,DAL,MEM,240:00,34.512358,81.973248,0.422038,4.821751,13.433265,...,0.217549,0.553944,1.860208,1.390167,2.562125,2.579181,10.837868,10.873075,1.062694,3.413240
3,20000070,2000-11-08,LAC,UTA,265:00,34.840361,71.230300,0.492003,4.290782,10.778424,...,0.410403,0.507130,1.610283,1.451214,2.825818,2.432686,12.436075,8.693520,4.559189,-2.760369
4,20000069,2000-11-08,DEN,MIN,265:00,38.792382,79.389939,0.489491,4.133178,9.077930,...,0.563591,0.737421,1.797572,1.962515,3.429119,3.099911,13.172195,11.273352,-0.934327,-2.911181


In [15]:
nba_training_df.drop(columns=['away_MIN', 'home_MIN'], inplace=True)

In [16]:
categorical_variables = nba_training_df.select_dtypes(include=['number']).columns
print(categorical_variables)

Index(['game_id', 'away_FGM', 'away_FGA', 'away_FG_PCT', 'away_FG3M',
       'away_FG3A', 'away_FG3_PCT', 'away_FTM', 'away_FTA', 'away_FT_PCT',
       ...
       'away_10g_player_BLK', 'home_10g_player_BLK', 'away_10g_player_TO',
       'home_10g_player_TO', 'away_10g_player_PF', 'home_10g_player_PF',
       'away_10g_player_PTS', 'home_10g_player_PTS',
       'away_10g_player_PLUS_MINUS', 'home_10g_player_PLUS_MINUS'],
      dtype='object', length=119)


In [17]:
nba_training_df['score_diff'] = nba_training_df['home_PTS_actual'] - nba_training_df['away_PTS_actual']
nba_training_df.drop(columns=['home_PTS_actual', 'away_PTS_actual'], inplace=True)
nba_training_df.head()
nba_training_df.to_csv('data/training_games4.csv', index=False)